# Proyecto Final: mythos-rag
---

**Curso:** Inteligencia Artificial Generativa con Grandes Modelos del Lenguaje
**Maestria en Informatica** - Escuela Colombiana de Ingenieria Julio Garavito

**Integrantes:**

- Hildebrando Pena Quezada (corpus e indice)
- Lancheros (pipeline rag)
- Ximena (evaluacion, interfaz y video)

**Repositorio:** https://github.com/brandoeci/mythos-rag

Un asistente de consulta sobre un corpus de mitologia griega de dominio publico
que responde citando el pasaje exacto, se abstiene cuando la respuesta no esta
en el corpus, y reporta cuando las fuentes se contradicen entre si.

### Problematica
---

Los LLM responden con seguridad sobre temas especificos, se equivocan, y nunca
dicen de donde sacaron el dato. En mitologia griega eso es peor de lo normal,
porque no existe *una* respuesta. Quien nace de quien, quien mato a quien, que
habia al principio del mundo: Hesiodo, Homero y Ovidio no coinciden, y no es que
uno este mal, es que son tradiciones distintas. Un LLM mezcla las versiones en
una sola respuesta que suena bien y no es de nadie.

A quien le sirve resolver esto: a cualquiera que tenga que **citar**. Un
estudiante de literatura clasica, alguien que prepara una clase, un traductor.
Para esa persona "Afrodita nacio de la espuma del mar" sin fuente no vale nada,
porque lo que necesita es poder escribir *Hesiodo, Teogonia* al lado y que sea
verdad.

Y no lo resuelve un Ctrl+F, por tres razones que medimos en este proyecto. La
primera es que el corpus esta en ingles y las preguntas en español: "afrodita"
aparece **cero** veces en las 861 mil palabras del corpus. La segunda es que las
obras se parten entre nombres griegos y latinos, asi que buscar "zeus" deja
afuera las 949 apariciones de "jove". La tercera es que la contradiccion esta
repartida entre dos obras distintas, y encontrarla exige leerlas las dos.

**Objetivo general.** Construir un asistente de consulta sobre un corpus de
mitologia griega de dominio publico que responda en lenguaje natural citando el
pasaje exacto, y que reconozca cuando el corpus no tiene la respuesta o la tiene
repetida de varias formas que no coinciden.

**Objetivos especificos.**

1. Armar un corpus citable de fuentes de dominio publico, donde cada fragmento
   sepa de que obra, de que autor y de que seccion salio, separando el texto del
   autor del que viene envuelto con el (prefacios, notas del traductor, indices).
2. Montar un retrieval hibrido, denso mas lexico, y medir con una tabla de
   ablacion cuanto aporta cada pieza en vez de afirmar que el sistema funciona.
3. Agregar dos comportamientos que un LLM suelto no tiene: abstenerse cuando el
   dato no esta, y reportar las dos versiones cuando las fuentes se contradicen.

### Trabajos Relacionados
---

La idea de darle al modelo un pasaje recuperado en vez de confiar en lo que
quedo en los pesos es de Lewis et al. (2020), que presentaron RAG justamente
para tareas donde hace falta conocimiento preciso y verificable. El retrieval
denso que usamos viene de esa linea: Karpukhin et al. (2020) mostraron con DPR
que embeddings entrenados para recuperacion le ganan a BM25 en preguntas
parafraseadas. BM25 en si es anterior y sigue vigente, formalizado por Robertson
y Zaragoza (2009), y la razon por la que lo mantuvimos es la que ellos mismos
explican: es exacto en terminos raros, y los nombres propios de la mitologia son
exactamente eso. Que convenga combinar los dos y no escoger uno tambien esta
estudiado; Cormack et al. (2009) mostraron que fusionar rankings funciona mejor
que cualquiera de los dos por separado.

Sobre abstenerse y revisar lo recuperado antes de responder, el trabajo mas
cercano es Self-RAG (Asai et al., 2023), donde el modelo critica sus propias
recuperaciones y decide si le sirven. Nuestro nodo `grade` es una version mucho
mas simple de esa idea: un umbral sobre la similitud, calibrado con las
preguntas que sabemos que no tienen respuesta en el corpus. Para los embeddings
multilingues usamos multilingual-E5 (Wang et al., 2024), que es lo que nos
permite preguntar en español sobre un corpus en ingles.

Donde no encontramos trabajo que nos sirviera directo fue en lo del conflicto.
Casi todo lo que hay asume que el corpus es **consistente**, y que si dos
pasajes no coinciden es un error que hay que resolver, por ejemplo escogiendo la
fuente mas confiable. Aca es lo contrario: la contradiccion no es ruido, es el
dato. Hesiodo y Homero discrepan sobre si Afrodita tiene madre y las dos
versiones son legitimas, asi que el sistema no debe elegir, debe mostrar las dos
con su cita. Lo mas parecido que vimos esta del lado de los grafos de
conocimiento, donde GraphRAG (Edge et al., 2024) usa la estructura para ver que
dicen varias fuentes del mismo nodo.

Y el punto de partida fueron los dos talleres del curso. En el taller 1 armamos
un modelo de n-gramas con smoothing, backoff, temperatura y penalizacion de
repeticion, que es el recordatorio de que un modelo de lenguaje sin fuentes
genera texto plausible y nada mas. En el taller 2 implementamos word2vec CBOW
(Mikolov et al., 2013) y medimos similitud coseno entre palabras, que es la
operacion sobre la que esta montado el indice denso de este proyecto. Por eso el
BM25 de aca tambien esta escrito a mano: es la forma de seguir entendiendo que
hace cada parte.

### Metodologia
---

#### Arquitectura

El pipeline es un grafo de LangGraph con una arista condicional:

| paso | nodo | que hace |
| --- | --- | --- |
| 1 | `retrieve` | busca en los dos indices y fusiona los rankings |
| 2 | `grade` | mide la confianza y decide si hay con que responder |
| 3a | `abstain` | si no, dice que el dato no esta y termina |
| 3b | `detect_conflict` | si si, revisa si las fuentes discrepan |
| 4 | `generate` | redacta en español y adjunta las citas |

La decision de diseño que sostiene todo el proyecto esta en el paso 4: **el
modelo nunca escribe la cita**. Se le pasan los pasajes numerados y solo
responde cuales uso; la cita la armamos nosotros desde el chunk, con el autor,
la obra y el locator que ya traia. Una cita inventada no queda improbable, queda
imposible. Y si redacta sin citar ninguno, la respuesta no pasa y el sistema se
abstiene.

#### Decisiones del stack

| pieza | que usamos | por que |
| --- | --- | --- |
| embeddings | `multilingual-e5-base`, local | las preguntas van en español y el corpus esta en ingles. local para no gastar cuota de API indexando |
| vector store | ChromaDB persistido | guarda la metadata junto al vector, que es lo que permite citar, y no hay que reindexar en cada corrida |
| lexico | BM25 escrito a mano | treinta lineas, una dependencia menos, y es lo que el curso ha venido haciendo |
| orquestacion | LangGraph | la abstencion es una arista condicional, no un `if` escondido en una funcion |
| generacion | Gemini 2.5 Flash | solo redacta; si no hay cuota el sistema responde de forma extractiva |
| interfaz | Gradio | la demo |

#### 1. El corpus

Ocho obras de dominio publico de Project Gutenberg. La seleccion no es por
cantidad sino porque **se contradicen entre si**: hay fuentes primarias
(Hesiodo, Homero, Apolonio, Ovidio) y dos compendios del siglo XIX (Bulfinch y
Berens) que cuentan los mismos mitos y discrepan en los detalles.

Apolodoro, que era una de las que queriamos, no esta en Gutenberg. Se reemplazo
por la Argonautica de Apolonio y el compendio de Berens.

In [ ]:
import pandas as pd

from corpus import load_corpus, summarize

corpus = load_corpus()
pd.DataFrame(summarize(corpus))

Verificar el corpus dio cuatro hallazgos que cambiaron el diseño. Estan
completos en el README; el que mas importa es que **tres de los ocho archivos
mezclan texto citable con texto que no lo es**:

- El ebook #348 no es la Teogonia, es una antologia de unas treinta obras, mas
  la introduccion de Evelyn-White. Los Himnos Homericos que trae son de Homero,
  no de Hesiodo. Citar el archivo entero como "Hesiodo, Teogonia" habria sido
  una cita falsa.
- Los dos tomos de Ovidio tienen cuatro capas y solo una es Ovidio: resumenes de
  libro, sinopsis por fabula, 109 notas de Riley del siglo XIX, y el verso.
- Bulfinch intercala citas de Milton y Byron en su propia prosa.

Por eso el chunking no puede recibir el texto crudo. Recibe secciones ya
atribuidas.

#### 2. De archivo a seccion citable

`iter_sections` resuelve el autor, la obra y el locator de cada tramo. Son
cuatro formatos de encabezado distintos, uno por tipo de archivo, y cada uno
tenia su propia trampa.

In [ ]:
from corpus import SOURCES, iter_sections

filas = []
for source in SOURCES:
    secciones = iter_sections(source, corpus[source])
    palabras = [len(s.text.split()) for s in secciones]
    filas.append({
        "archivo": source.filename,
        "secciones": len(secciones),
        "palabras_min": min(palabras),
        "palabras_max": max(palabras),
        "ejemplo_locator": secciones[0].locator,
    })

pd.DataFrame(filas)

Partir los archivos dejo ver cuatro cosas que leerlos no habia mostrado, y las
cuatro son la misma: el texto del autor viene envuelto en texto que no es del
autor, y lo que lo envuelve es corto y denso en nombres propios, que es justo lo
que el retrieval premia.

1. **Bulfinch tenia cada capitulo dos veces**, porque su tabla de contenido usa
   los mismos encabezados que la obra. Quedaban 24 secciones citables de 25 a 65
   palabras, puros nombres seguidos.
2. **El capitulo II estaba escrito `CHAPTER II`** y los otros 57 `Chapter`. El
   patron no lo veia, asi que el capitulo II completo, que es *Prometheus and
   Pandora*, quedaba pegado al I: 11.203 palabras contra las 4.643 que tiene.
   Pandora es una de nuestras preguntas, o sea que citaba el capitulo
   equivocado.
3. **Berens cierra con un anuncio de la editorial** de 1.415 palabras sobre un
   curso de ingles, despues de 19 secciones de indice de pronunciacion.
4. **Las notas de Berens estaban citadas como ese anuncio**, porque van al final
   del archivo, despues del indice, y caian bajo su encabezado. Son texto propio
   de Berens y de ellas dependen dos preguntas verificadas, entre ellas un lado
   de la contradiccion sobre quien era la esposa de Hefesto. Quedaron en su
   propia seccion, con locator `notas`.

La leccion del metodo: ninguna de las cuatro se asomo leyendo los archivos. La 1
y la 2 salieron porque el chunking tiro **23 ids repetidos**, y la 4 porque
verificamos que cada pasaje de oro exista de verdad en algun chunk. Son chequeos
baratos que conviene dejar corriendo, no revisiones de una sola vez.

#### 3. Chunking

Ventana de 220 palabras con 50 de solape, sobre las secciones y nunca sobre el
archivo. El solape existe para que una frase partida por la mitad siga siendo
recuperable desde los dos lados. Cada chunk se queda con el autor, la obra y el
locator de su seccion, que es lo que despues se imprime en la cita.

In [ ]:
from index import build_chunks

chunks = build_chunks(corpus)
print(f"chunks: {len(chunks)}")
print(f"ids unicos: {len(set(c.id for c in chunks))}")

palabras = [len(c.text.split()) for c in chunks]
print(f"palabras por chunk: min {min(palabras)}, max {max(palabras)}")
print()
print("asi queda una cita:")
print(f"   {chunks[0].author}, {chunks[0].work} ({chunks[0].locator})")

#### 4. El indice lexico

BM25 escrito a mano. El score de un termino en un documento es

$$
\text{idf}(t)\cdot\frac{f(t,d)\,(k_1+1)}
{f(t,d)+k_1\left(1-b+b\,\frac{|d|}{\text{avgdl}}\right)}
$$

con $k_1 = 1.5$ y $b = 0.75$. El `idf` castiga los terminos que estan en todas
partes y el denominador normaliza por el largo del documento, que es lo que le
falta a un conteo pelado.

Y aca aparecio el problema que casi hundio la mitad del sistema: el corpus esta
en ingles y las preguntas en español.

In [ ]:
import re

from index import Bm25Index, tokenize_query

lexico = Bm25Index(chunks)
print(f"vocabulario: {len(lexico.idf)} terminos, largo promedio {lexico.avgdl:.1f}")
print()

# las formas en español no aparecen ni una vez. las griegas y latinas si, y la
# latina es la mas frecuente porque butler traduce con los nombres romanos
texto = " ".join(c.text for c in chunks).lower()
for nombre in ["afrodita", "aphrodite", "venus", "zeus", "jove", "hefesto", "vulcan"]:
    print(f"   {nombre:12} {len(re.findall(chr(92) + 'b' + nombre + chr(92) + 'b', texto)):>5}")

In [ ]:
# la tabla de nombres se aplica solo a la consulta, nunca a los documentos
pregunta = "¿Quien era la madre de Afrodita?"
print("consulta expandida:", tokenize_query(pregunta))
print()
for chunk, score in lexico.search(pregunta, k=3):
    print(f"   {score:6.2f}  {chunk.author}, {chunk.work} ({chunk.locator})")

#### 5. El indice denso

`multilingual-e5-base`, calculado localmente. Los prefijos `query:` y
`passage:` son asimetricos a proposito: el modelo fue entrenado asi y sin ellos
la similitud baja. Son 4.674 chunks y en CPU toma alrededor de 25 minutos, asi
que el indice queda persistido en `index/` y en las corridas siguientes se lee
del disco.

In [ ]:
from index import DenseIndex

denso = DenseIndex(chunks)
denso.build(chunks)

for chunk, similitud in denso.search(pregunta, k=3):
    print(f"   {similitud:.3f}  {chunk.author}, {chunk.work} ({chunk.locator})")

#### 6. El hibrido

Los dos rankings se normalizan a [0, 1] y se mezclan con `alpha`. Hace falta
normalizar porque no viven en la misma escala: el BM25 de una pregunta con
nombres propios pasa de 20 y el coseno nunca pasa de 1.

`alpha = 1.0` es solo denso y `alpha = 0.0` solo lexico, asi que las filas de la
tabla de ablacion son el mismo codigo con un parametro distinto.

In [ ]:
from index import HybridRetriever

for alpha in (1.0, 0.5, 0.0):
    print(f"alpha = {alpha}")
    for chunk, score in HybridRetriever(denso, lexico, alpha=alpha).search(pregunta, k=3):
        print(f"   {score:.3f}  {chunk.author}, {chunk.work} ({chunk.locator})")
    print()

#### 7. El grafo

La abstencion es una arista condicional del grafo y no un `if` escondido en una
funcion, que es lo que permite decir que el sistema tiene dos caminos y
mostrarlos.

In [ ]:
from rag import build_graph

grafo = build_graph()
print("nodos:")
for nodo in grafo.get_graph().nodes:
    print(" -", nodo)
print()
print("aristas:")
for arista in grafo.get_graph().edges:
    marca = "  (condicional)" if arista.conditional else ""
    print(f"   {arista.source} -> {arista.target}{marca}")

#### 8. Calibrar la abstencion

El umbral de `grade` no se puso a ojo. Las 10 preguntas del grupo `fuera` son
preguntas cuya respuesta sabemos que **no** esta en el corpus (Gilgamesh,
Cuchulainn, Mitra, Arquimedes), asi que la confianza del retrieval en ese grupo
es la que hay que dejar por debajo del umbral.

Importante: la confianza **no** puede salir del score fusionado, porque la
fusion normaliza cada ranking por consulta y el primer resultado sale siempre en
1.0 aunque sea basura. El coseno del indice denso si es absoluto y comparable
entre preguntas.

In [ ]:
from evaluate import calibracion

tabla = calibracion()
tabla.groupby("grupo")["confianza"].describe()[["count", "mean", "min", "max"]]

**El resultado fue que no sirve**, y vale la pena dejarlo escrito porque es la
medicion que cambio el diseño:

| grupo | min | media | max |
| --- | --- | --- | --- |
| directa | 0.789 | 0.803 | 0.826 |
| conflicto | 0.768 | 0.794 | 0.825 |
| fuera | 0.766 | 0.788 | 0.805 |

Las tres distribuciones se superponen casi del todo. El grupo `fuera` llega a
0.805 y `directa` arranca en 0.789, asi que **no existe ningun umbral que los
separe**: con 0.806 el sistema se abstiene de 8 de las 10 preguntas que si
tienen respuesta, y con 0.789 solo atrapa 3 de las 10 que no la tienen. Las
similitudes de E5 viven en una banda angosta, y "quien acompaño a Gilgamesh"
igual encuentra pasajes sobre el viaje de un heroe que se le parecen.

La señal que si funciona sale del indice **lexico**, que es lo contrario de lo
que esperabamos. Las preguntas del grupo `fuera` nombran entidades que no estan
en el corpus, y para un termino ausente BM25 devuelve cero exacto, no un numero
pequeño.

La condicion no es "el BM25 dio cero" sino **"la pregunta nombra entidades y
ninguna aparece en el corpus"**. La diferencia importa: "¿quien creo a la
humanidad?" y "¿que habia al principio del mundo?" tambien dan cero, porque no
tienen ningun nombre propio, y esas dos si tienen respuesta. Sin esa distincion
el sistema se abstendria de dos preguntas buenas.

In [ ]:
from rag import entidades_de_la_pregunta, sin_evidencia_lexica

from evaluate import load_questions

for q in load_questions():
    if q.group == "fuera":
        marca = "se abstiene" if sin_evidencia_lexica(q.question) else "responde   "
        nombres = [e[0] for e in entidades_de_la_pregunta(q.question)]
        print(f"{marca}  {str(nombres):34} {q.question}")

Sale 7 de 10, y 0 falsas abstenciones sobre las 20 contestables. Las tres que
no atrapa son las de la segunda clase del grupo: la entidad **si** esta en el
corpus y lo que falta es el dato puntual. El caballo de Troya aparece en 19
chunks de seis obras y nunca con una medida; Hefesto y el escudo de Aquiles
estan descritos en detalle pero sin cuanto tardo la forja. Para esas tres no
hay señal lexica posible, porque los terminos estan. Resolverlas pide que algo
**lea** el pasaje y concluya que no responde, y eso es el juez con LLM del nodo
`grade`, que es donde se parece a Self-RAG.

### Resultados y Discusion
---

Cada fila cambia una sola cosa respecto a la anterior, para ver que aporto cada
pieza en vez de afirmar que el sistema funciona. Los denominadores: 20 preguntas
contestables (10 de respuesta directa y 10 de conflicto), 10 de conflicto y 10
de fuera del corpus.

La columna que importa es `pasaje_de_oro_20`. Al verificar las 30 preguntas a
mano copiamos los pasajes verbatim que las respaldan, asi que se puede comprobar
directo si el retrieval recupero el pasaje bueno, sin juez y sin gastar cuota.
Se compara por secuencias de 8 palabras, porque un pasaje de oro puede caer
partido entre dos chunks.

In [ ]:
from evaluate import ablation_table

ablation_table()

Lo que corrio, sin cuota de API:

| configuracion | pasaje_de_oro_20 | citas_20 | conflictos_10 | falsos_10 | se_abstuvo_10 |
| --- | --- | --- | --- | --- | --- |
| rag_lexico | 6 | 8 | 0 | 0 | 7 |
| rag_denso | 9 | 9 | 0 | 0 | 7 |
| rag_hibrido | **11** | 9 | 0 | 0 | 7 |
| rag_hibrido_conflictos | **11** | 9 | **5** | 1 | 7 |

**El hibrido le gana a los dos indices por separado**, que era la razon de
tenerlos a los dos: 11 contra 9 del denso y 6 del lexico. No es que uno sea
mejor, es que fallan en preguntas distintas. El lexico no puede con "¿quien creo
a la humanidad?", que no tiene un solo nombre propio; el denso no puede con
"Absirto", porque un nombre raro no tiene vecindario semantico que lo ubique.

Hay que mirar como se comporta eso al mover `alpha` y `k`, porque los valores
por defecto no se pueden dejar puestos a dedo.

In [ ]:
from index import Bm25Index, DenseIndex, HybridRetriever
from evaluate import load_questions, pasaje_recuperado

contestables = [q for q in load_questions() if not q.should_abstain]

filas = []
for alpha in (0.0, 0.3, 0.5, 0.7, 1.0):
    retriever = HybridRetriever(denso, lexico, alpha=alpha)
    fila = {"alpha": alpha}
    for k in (6, 10, 20):
        fila[f"k={k}"] = sum(
            pasaje_recuperado(q, [c for c, _ in retriever.search(q.question, k=k)])
            for q in contestables
        )
    filas.append(fila)

pd.DataFrame(filas)

| alpha | k=6 | k=10 | k=20 |
| --- | --- | --- | --- |
| 0.0 (solo lexico) | 6 | 10 | 11 |
| 0.3 | 9 | 11 | 13 |
| 0.5 | **11** | 12 | 13 |
| 0.7 | 10 | 11 | **14** |
| 1.0 (solo denso) | 9 | 9 | **14** |

Esto matiza el resultado anterior y es lo mas interesante que salio: **la
ventaja del hibrido esta concentrada en `k` pequeño**. Con `k = 6` la mezcla
mitad y mitad saca 11 y el denso solo saca 9; con `k = 20` el denso solo ya
empata en 14. O sea que el hibrido no recupera cosas que el denso no pueda
recuperar nunca, las recupera **mas arriba**.

Y `k` pequeño es justamente el regimen que importa, porque en el prompt solo
caben unos pocos pasajes. Por eso `alpha = 0.5` y `TOP_K = 6` quedaron como
valores por defecto: son los mejores en el unico punto de operacion que el
generador puede usar.

#### Lo que no funciona

Hay que decirlo con los numeros al lado.

**El pasaje de oro se recupera en 11 de 20.** Es la mitad, y es el techo de todo
lo que viene despues: en las otras 9 el generador no tiene con que acertar
aunque redacte bien. Partir el corpus en chunks de 220 palabras con solape de 50
puede estar cortando justo donde hace falta, y la cifra de `k = 20` dice que el
pasaje suele estar cerca pero no en los primeros seis.

**El detector de conflictos va 5 de 10, con 1 falso positivo.** La primera
version marcaba conflicto en 8 de las 10 preguntas de respuesta directa, porque
comparaba todos los nombres propios de los pasajes. Se arreglo con dos
condiciones: que la oracion contenga la **relacion** que se pregunta (madre,
esposa, creo) traducida al ingles, y que los nombres se tomen solo de la
**ventana** alrededor de esa palabra. Quedo del lado conservador a proposito:
preferimos no reportar un conflicto que no existe antes que encontrarlos todos,
porque un conflicto inventado es del mismo tipo de error que una cita falsa.

**Tres de las diez preguntas fuera del corpus no se atrapan**, y son las de la
segunda clase, donde la entidad esta y falta el dato. Para esas no hay señal
lexica posible.

**La fila de control no se pudo correr** por falta de `GOOGLE_API_KEY`, y la
columna del juez quedo vacia. Se omiten en vez de llenarlas de ceros, que se
leerian como un resultado medido. Lo que si quedo demostrado es que el sistema
corre completo sin cuota, en modo extractivo.

### Conclusion
---

El sistema responde citando el pasaje exacto, se abstiene en 7 de las 10
preguntas cuya respuesta no esta en el corpus sin equivocarse ni una vez sobre
las 20 que si tienen respuesta, y reporta 5 de las 10 contradicciones con un
falso positivo. El retrieval hibrido recupera el pasaje correcto en 11 de 20,
contra 9 del denso solo y 6 del lexico solo.

Si tuvieramos que quedarnos con una sola idea del proyecto es esta: **el modelo
nunca escribe la cita**. Solo dice que pasaje uso, y la cita la armamos nosotros
desde el chunk, con el autor, la obra y el locator que ya traia. Eso convierte
la cita falsa de un problema de probabilidad en uno de imposibilidad, y es lo
que permitio despues gastar el tiempo en lo que si decide la calidad, que es el
retrieval.

Lo que mas nos sorprendio fue donde estaba el trabajo de verdad. Esperabamos que
fuera el pipeline, y fue el corpus. Y casi todo lo que encontramos ahi es del
mismo tipo: **el texto termina firmado por quien no lo escribio**. El ebook de
Hesiodo es una antologia de treinta obras y los Himnos Homericos que trae son de
Homero; de los dos tomos de Ovidio, solo una de cuatro capas es Ovidio; Bulfinch
tenia su tabla de contenido indexada como si fuera la obra y su capitulo II
pegado al I por una mayuscula; las notas de Berens quedaban citadas como un
anuncio de la editorial; y dos obras estaban firmadas por Hesiodo porque el
codigo las comparaba contra titulos que no existen en el archivo. Ninguno se veia
leyendo. Todos salieron de chequeos baratos: que los ids sean unicos, que cada
pasaje de oro exista en algun chunk, listar autor y obra y leer la lista.

El segundo aprendizaje es metodologico y nos costo dos preguntas del set de
evaluacion. Habiamos verificado que Hiperborea y las colas de Cerbero no estaban
en el corpus, y las dos verificaciones estaban mal **por la misma razon**: se
busco una sola forma de la palabra. "hyperborea" da 0 pero "hyperboreans" da 6,
y "tails" da 0 pero Berens dice "the tail of a dragon". Es la misma familia que
el error de contar substrings en vez de palabras que ya habiamos documentado.
Las dos preguntas se reemplazaron por negativos verificados con todas las
variantes, y uno de los dos es deliberadamente dificil: el corpus **si** trae
mitologia nordica (Loki 59 veces, Valhalla 18, Balder 10), pero no el ciclo de
los Volsungos, asi que preguntar como murio Sigurd castiga responder por
cercania tematica.

Y el tercero es el que mas nos cambio el diseño. Queriamos abstenernos con la
similitud del coseno y **no se puede**: medidas sobre los tres grupos, las
distribuciones se superponen y no hay umbral que las separe. La señal util
estaba en el indice lexico, que es el que uno pondria para lo contrario. Un
termino que no esta en el corpus da cero exacto, no un numero pequeño, y eso si
es una decision. Los dos indices no solo se complementan para recuperar: se
complementan para **saber cuando callarse**.

Lo que sigue, en orden: subir el pasaje de oro de 11 de 20, que es el techo de
todo lo demas; poner el juez con LLM en `grade` para las tres preguntas que la
señal lexica no puede atrapar; y subir el recall del detector de conflictos sin
perder la precision.